In [ ]:
# Imports and configuration
import os
import math
import json
import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML

# Backend base URL
backendUrl = os.getenv('CASE_MGMT_BACKEND_URL', 'http://localhost:3000')

# Fallback counterparty ID (verified working one)
FALLBACK_CP_ID = 'dbtrAcct_de52b2c4041c4a03a73a92cfb0b8cf35'

# Counterparty to visualize
counterpartyId = os.getenv('COUNTERPARTY_ID', FALLBACK_CP_ID)

# print(f'Using backend: {backendUrl}')
# print(f'Using counterpartyId: {counterpartyId}')





In [ ]:
# Fetch counterparty network analysis data
def fetch_counterparty_network(cp_id, backend_url):
    url = f"{backend_url}/api/v1/lakehouse/network-analysis/counterparty/{cp_id}"
    params = {'tenantId': 'DEFAULT'}
    try:
        resp = requests.get(url, params=params, timeout=15)
        if resp.status_code == 404 and cp_id != FALLBACK_CP_ID:
            print(f'Network data not found for {cp_id}, trying fallback: {FALLBACK_CP_ID}')
            url = f"{backend_url}/api/v1/lakehouse/network-analysis/counterparty/{FALLBACK_CP_ID}"
            resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print('Error fetching counterparty network:', e)
        return None

data = fetch_counterparty_network(counterpartyId, backendUrl)
if not data:
    display(HTML(f"<div style='color:#ef4444'>No network data available for counterparty: {counterpartyId}</div>"))
# else:
#     print('Fetched network data keys:', list(data.keys()))






In [ ]:
# Normalise network payloads
# API returns: { centerCounterparty, counterparties, edges, ... }
nodes = []
edges = []
if data and isinstance(data, dict):
    # Build nodes array from centerCounterparty and counterparties
    center = data.get('centerCounterparty')
    if center and isinstance(center, dict):
        nodes.append(center)
    
    counterparties = data.get('counterparties', [])
    if isinstance(counterparties, list):
        nodes.extend([cp for cp in counterparties if isinstance(cp, dict)])
    
    # Get edges directly from top level
    edges = data.get('edges', [])
    if isinstance(edges, list):
        edges = [e for e in edges if isinstance(e, dict)]

# Standardize node keys for downstream usage
normalized_nodes = []
for n in nodes:
    if not isinstance(n, dict): continue
    # Map API keys to expected keys
    new_node = n.copy()
    if 'counterpartyId' in n:
        new_node['id'] = n['counterpartyId']
    if 'counterpartyName' in n:
        new_node['label'] = n['counterpartyName']
    normalized_nodes.append(new_node)
nodes = normalized_nodes

# print(f'Nodes: {len(nodes)}, Edges: {len(edges)}')

# Build layout and render with Plotly
def build_positions(nodes):
    positions = {}
    n = max(len(nodes), 1)
    center_idx = 0
    for i, node in enumerate(nodes):
        if node.get('id') == counterpartyId:
            center_idx = i
            break

    radius = 300
    angle_step = 2 * math.pi / max(n - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1
    return positions

positions = build_positions(nodes)
edge_x, edge_y = [], []
for e in edges:
    s = e.get('source') or e.get('from')
    t = e.get('target') or e.get('to')
    if s in positions and t in positions:
        x0, y0 = positions[s]
        x1, y1 = positions[t]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=1, color='#cbd5e1'), hoverinfo='none')
node_x, node_y, node_text, marker_size, marker_color, node_ids = [], [], [], [], [], []

def color_for(node):
    status = node.get('status', 'normal')
    if status in ('alert', 'flagged'): return '#ef4444'
    if status == 'investigation': return '#f59e0b'
    return '#6366f1' if node.get('type') == 'COUNTERPARTY' else '#3b82f6'

for n in nodes:
    nid = n.get('id'); x, y = positions.get(nid, (0,0))
    node_x.append(x); node_y.append(y)
    label = n.get('label') or nid
    node_text.append(f"{label} ({nid})")
    marker_size.append(32 if nid == counterpartyId else 22)
    marker_color.append(color_for(n)); node_ids.append(nid)

node_trace = go.Scatter(x=node_x, y=node_y, mode='markers+text', text=[t.split(' ')[0] for t in node_text],
    hovertext=node_text, hoverinfo='text', customdata=node_ids,
    marker=dict(showscale=False, color=marker_color, size=marker_size, line_width=3, line_color='#ffffff'),
    textposition='bottom center')

fig = go.Figure(data=[edge_trace, node_trace], layout=go.Layout(showlegend=False, hovermode='closest',
    margin=dict(b=20,l=5,r=5,t=20), xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False), height=600,
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)'))

if nodes:
    graph_id = 'network-graph-id'
    graph_html = pio.to_html(fig, full_html=False, config={'displayModeBar': False}, div_id=graph_id)
    
    sidebar_html = """
    <div id="sidebar-container" style="width: 250px; background: #ffffff; color: #0f172a; padding: 14px; border-radius: 14px; box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1); font-family: 'Inter', system-ui, sans-serif; margin-left: 14px; border: 1px solid #e2e8f0; display: flex; flex-direction: column; height: 600px; overflow-y: auto; transition: all 0.3s ease;">
        <div id="view-overview">
            <h2 style="margin: 0 0 20px 0; color: #0f172a; font-size: 1.25rem; font-weight: 700; border-bottom: 1px solid #e2e8f0; padding-bottom: 12px;">Counterparty Overview</h2>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Counterparty ID</div>
                <div id="overview-id" style="font-weight: 600; font-size: 1rem; color: #0f172a;">-</div>
            </div>

            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Name</div>
                <div id="overview-name" style="font-weight: 600; font-size: 1.1rem; color: #0284c7;">-</div>
            </div>

            <div style="margin-bottom: 14px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Relationship</div>
                <div id="overview-type" style="display: inline-block; background: #f1f5f9; color: #475569; padding: 4px 12px; border-radius: 9999px; font-size: 0.75rem; margin-top: 4px; font-weight: 500;">-</div>
            </div>

            <div style="background: #f8fafc; padding: 20px; border-radius: 20px; border: 1px solid #e2e8f0;">
                <h3 style="margin-top: 0; font-size: 0.85rem; color: #64748b; margin-bottom: 16px; text-transform: uppercase; font-weight: 600;">Network Summary</h3>
                <div style="display: grid; grid-template-columns: 1fr; gap: 16px;">
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Linked Accounts</span>
                        <span id="overview-linked" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Total Transactions</span>
                        <span id="overview-total-tx" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Total Value</span>
                        <span id="overview-total-value" style="font-weight: 600; color: #059669;">$0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Entities with Alerts</span>
                        <span id="overview-alerts" style="font-weight: 600; color: #dc2626;">0</span>
                    </div>
                </div>
            </div>
        </div>
        <div id="view-details" style="display: none;">
            <h2 style="margin: 0 0 20px 0; color: #0f172a; font-size: 1.25rem; font-weight: 700; border-bottom: 1px solid #e2e8f0; padding-bottom: 12px;">Entity Details</h2>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Entity ID</div>
                <div id="side-id" style="font-weight: 600; font-size: 1rem; color: #0f172a; word-break: break-all;">-</div>
            </div>

            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Name</div>
                <div id="side-name" style="font-weight: 600; font-size: 1.1rem; color: #d97706;">-</div>
            </div>
            <div style="margin-bottom: 14px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Type</div>
                <div id="side-relation" style="display: inline-block; background: #fff7ed; color: #9a3412; padding: 4px 12px; border-radius: 9999px; font-size: 0.75rem; margin-top: 4px; font-weight: 500;">Counterparty</div>
            </div>
            <div style="background: #f8fafc; padding: 20px; border-radius: 20px; border: 1px solid #e2e8f0;">
                <h3 style="margin-top: 0; font-size: 0.85rem; color: #64748b; margin-bottom: 16px; text-transform: uppercase; font-weight: 600;">Transaction Statistics</h3>
                <div style="display: grid; grid-template-columns: 1fr; gap: 16px;">
                    <div style="display: flex; justify-content: space-between;"><span style="color: #64748b; font-size: 0.8rem;">Transactions</span><span id="side-tx" style="font-weight: 600; color: #0f172a;">0</span></div>
                    <div style="display: flex; justify-content: space-between;"><span style="color: #64748b; font-size: 0.8rem;">Total Value</span><span id="side-value" style="font-weight: 600; color: #059669;">$0</span></div>
                    <div style="display: flex; justify-content: space-between;"><span style="color: #64748b; font-size: 0.8rem;">Velocity</span><span id="side-velocity" style="font-weight: 600; color: #dc2626;">HIGH</span></div>
                </div>
            </div>
            <div style="margin-top: 14px; text-align: center;"><button onclick="resetSidebar()" style="background: white; border: 1px solid #cbd5e1; color: #64748b; padding: 8px 16px; border-radius: 8px; font-size: 0.75rem; cursor: pointer; font-weight: 500; transition: all 0.2s;">Back to Overview</button></div>
        </div>
    </div>
    """
    
    cp_details_json = json.dumps(data.get('centerCounterparty', {}))
    interactive_script = f"""
    <script>
    (function() {{
        const nodes = {json.dumps(nodes)};
        const edges = {json.dumps(edges)};
        const cpDetails = {cp_details_json};
        const graphDiv = document.getElementById('{graph_id}');
        
        function initializeOverview() {{
            document.getElementById('overview-id').innerText = cpDetails.id || '-';
            document.getElementById('overview-name').innerText = cpDetails.name || '-';
            document.getElementById('overview-type').innerText = cpDetails.type || 'Business Entity';
            document.getElementById('overview-linked').innerText = Math.max(nodes.length - 1, 0);
            let txS = 0, valS = 0, alertsS = 0;
            edges.forEach(e => {{ txS += e.txCount || 0; valS += e.totalAmount || 0; }});
            nodes.forEach(n => {{ if (n.flags && n.flags.alerted) alertsS++; }});
            document.getElementById('overview-total-tx').innerText = txS;
            document.getElementById('overview-total-value').innerText = '$' + valS.toLocaleString(undefined, {{minimumFractionDigits: 2}});
            document.getElementById('overview-alerts').innerText = alertsS;
        }}
        
        window.resetSidebar = function() {{
            document.getElementById('view-overview').style.display = 'block';
            document.getElementById('view-details').style.display = 'none';
            document.getElementById('sidebar-container').style.background = '#ffffff';
            initializeOverview();
        }};
        
        function updateSidebar(nodeId) {{
            const node = nodes.find(n => n.id === nodeId);
            if (!node) return;
            document.getElementById('view-overview').style.display = 'none';
            document.getElementById('view-details').style.display = 'block';
            document.getElementById('sidebar-container').style.background = '#ffffff';
            
            const linkedEdges = edges.filter(e => e.source === nodeId || e.target === nodeId || e.from === nodeId || e.to === nodeId);
            let txC = 0, valC = 0, holder = node.label || node.id, rel = 'Connected Entity';
            
            if (nodeId === cpDetails.id) {{
                holder = cpDetails.name || holder; txC = cpDetails.transactions || 0;
                valC = cpDetails.totalValue || 0; rel = 'Primary Counterparty';
            }} else {{
                linkedEdges.forEach(e => {{ txC += e.txCount || 0; valC += e.totalAmount || 0; }});
            }}
            document.getElementById('side-id').innerText = node.id;
            document.getElementById('side-name').innerText = holder;
            document.getElementById('side-tx').innerText = txC;
            document.getElementById('side-value').innerText = '$' + valC.toLocaleString(undefined, {{minimumFractionDigits: 2}});
            document.getElementById('side-velocity').innerText = txC > 10 ? 'HIGH' : 'LOW';
        }}
        
        setTimeout(() => initializeOverview(), 500);
        if (graphDiv) {{
            graphDiv.on('plotly_click', d => {{ if (d.points && d.points[0].customdata) updateSidebar(d.points[0].customdata); }});
            graphDiv.on('plotly_doubleclick', resetSidebar);
        }}
    }})();
    </script>
    """
    
    # TEMPLATE BUG FIX: Use {graph_html} instead of {{graph_html}}
    container_html = f"""
    <div style="display: flex; align-items: stretch; background-color: #ffffff; padding: 0px; gap: 0px;">
        <div style="flex: 1; background: #f8fafc; border-radius: 14px; border: 1px solid #e2e8f0; overflow: hidden; position: relative;">
            {graph_html}
        </div>
        {sidebar_html}
    </div>
    {interactive_script}
    """
    display(HTML(container_html))
else:
    display(HTML('<div>No nodes to display</div>'))


